# 06 Model Comparison

Bayesian comparison between:

- Single Sérsic model
- Double Sérsic model

The mock galaxy is generated from a DoubleSersicModel and fitted with both models.
The comparison is performed using the Dynesty Bayesian evidence.


In [ ]:
from baymorph import (
    ConvolvedModel,
    DoubleSersicModel,
    DynestyFitter,
    GaussianPSF,
    Parameter,
    ParameterSet,
    SersicModel,
    add_gaussian_noise,
    make_constant_noise_map,
    show_log_image,
)

import numpy as np

## Generate true double-Sérsic galaxy

In [ ]:
bulge = SersicModel(
    flux=3000,
    re=6,
    n=4,
    q=0.8,
    theta=0.0,
    x0=30,
    y0=30,
)

disk = SersicModel(
    flux=7000,
    re=12,
    n=1,
    q=0.8,
    theta=0.0,
    x0=30,
    y0=30,
)

true_model = DoubleSersicModel(
    bulge=bulge,
    disk=disk,
)

In [ ]:
psf = GaussianPSF(
    fwhm=3.0,
)

true_model = ConvolvedModel(
    true_model,
    psf,
)

In [ ]:
clean_image = true_model.render(
    shape=(60, 60),
)

data = add_gaussian_noise(
    clean_image,
    sigma=1.0,
    seed=42,
)

noise = make_constant_noise_map(
    data.shape,
    sigma=1.0,
)

In [ ]:
show_log_image(
    data,
    title="True double-Sersic galaxy",
)

## Single Sérsic fit

In [ ]:
def single_model_factory(
    flux,
    re,
    n,
):
    model = SersicModel(
        flux=flux,
        re=re,
        n=n,
        q=0.8,
        theta=0.0,
        x0=30,
        y0=30,
    )

    return ConvolvedModel(
        model,
        psf,
    )

In [ ]:
single_parameters = ParameterSet(
    [
        Parameter(
            "flux",
            1000,
            15000,
        ),
        Parameter(
            "re",
            1,
            25,
        ),
        Parameter(
            "n",
            0.5,
            8,
        ),
    ]
)

In [ ]:
single_fitter = DynestyFitter(
    model_factory=single_model_factory,
    data=data,
    noise=noise,
    parameters=single_parameters,
)

In [ ]:
single_result = single_fitter.run(
    nlive=150,
    sample="rwalk",
)

In [ ]:
single_logz = single_result.logz[-1]

print(
    "Single Sérsic logZ:",
    single_logz,
)

## Double Sérsic fit

In [ ]:
def double_model_factory(
    flux_bulge,
    re_bulge,
    flux_disk,
    re_disk,
):
    bulge = SersicModel(
        flux=flux_bulge,
        re=re_bulge,
        n=4,
        q=0.8,
        theta=0.0,
        x0=30,
        y0=30,
    )

    disk = SersicModel(
        flux=flux_disk,
        re=re_disk,
        n=1,
        q=0.8,
        theta=0.0,
        x0=30,
        y0=30,
    )

    model = DoubleSersicModel(
        bulge=bulge,
        disk=disk,
    )

    return ConvolvedModel(
        model,
        psf,
    )

In [ ]:
double_parameters = ParameterSet(
    [
        Parameter(
            "flux_bulge",
            500,
            6000,
        ),
        Parameter(
            "re_bulge",
            1,
            8,
        ),
        Parameter(
            "flux_disk",
            3000,
            12000,
        ),
        Parameter(
            "re_disk",
            5,
            20,
        ),
    ]
)

In [ ]:
double_fitter = DynestyFitter(
    model_factory=double_model_factory,
    data=data,
    noise=noise,
    parameters=double_parameters,
)

In [ ]:
double_result = double_fitter.run(
    nlive=150,
    sample="rwalk",
)

In [ ]:
double_logz = double_result.logz[-1]

print(
    "Double Sérsic logZ:",
    double_logz,
)

## Model comparison

In [ ]:
delta_logz = (
    double_logz
    - single_logz
)

print(
    "ΔlogZ:",
    delta_logz,
)

In [ ]:
if delta_logz < 1:
    print(
        "Inconclusive"
    )

elif delta_logz < 3:
    print(
        "Weak evidence"
    )

elif delta_logz < 5:
    print(
        "Moderate evidence"
    )

else:
    print(
        "Strong evidence for Double Sérsic"
    )

In [ ]:
print(
    "BayMorph correctly prefers the generating model."
)